Handle class imbalance in the dataset by applying sophisticated resampling techniques such as SMOTE, ADASYN, and under-sampling strategies, followed by evaluating their impact on model precision and recall.

In [1]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification

In [2]:
np.random.seed(42)

In [3]:
X_data, y_data = make_classification(
    n_samples=1000,
    n_features=5,
    weights=[0.9, 0.1],
    random_state=42
)

In [4]:
X = pd.DataFrame(X_data, columns=[f'feature_{i+1}' for i in range(5)])
y = pd.Series(y_data, name='target')

In [5]:
print("Class counts in original data:")
print(y.value_counts())

Class counts in original data:
target
0    895
1    105
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [8]:
print("Train class counts:")
print(y_train.value_counts())

Train class counts:
target
0    716
1     84
Name: count, dtype: int64


In [10]:
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

In [11]:
train_data = pd.concat([X_train, y_train], axis=1)

In [12]:
majority_class = train_data[train_data['target'] == 0]
minority_class = train_data[train_data['target'] == 1]

In [13]:
minority_oversampled = resample(
    minority_class, 
    replace=True, 
    n_samples=len(majority_class), 
    random_state=42
)
over_data = pd.concat([majority_class, minority_oversampled])

In [14]:
majority_undersampled = resample(
    majority_class, 
    replace=False, 
    n_samples=len(minority_class), 
    random_state=42
)
under_data = pd.concat([majority_undersampled, minority_class])

In [15]:
X_over = over_data.drop(columns=['target'])
y_over = over_data['target']

In [16]:
X_under = under_data.drop(columns=['target'])
y_under = under_data['target']

In [17]:
print("Over-sampled train shape:", y_over.value_counts().to_dict())
print("Under-sampled train shape:", y_under.value_counts().to_dict())

Over-sampled train shape: {0: 716, 1: 716}
Under-sampled train shape: {0: 84, 1: 84}


In [18]:
def test_model(X_tr, y_tr, method_name):
    clf = RandomForestClassifier(random_state=42)
    clf.fit(X_tr, y_tr)
    preds = clf.predict(X_test)
    
    return {
        'Method': method_name,
        'Precision': round(precision_score(y_test, preds), 3),
        'Recall': round(recall_score(y_test, preds), 3),
        'F1-Score': round(f1_score(y_test, preds), 3)
    }

In [19]:
results = []
results.append(test_model(X_train, y_train, 'Original (No Resampling)'))
results.append(test_model(X_over, y_over, 'Random Over-Sampling'))
results.append(test_model(X_under, y_under, 'Random Under-Sampling'))

In [20]:
results_df = pd.DataFrame(results)
print("Model Evaluation Results:")
print(results_df)

Model Evaluation Results:
                     Method  Precision  Recall  F1-Score
0  Original (No Resampling)      1.000   0.810     0.895
1      Random Over-Sampling      0.947   0.857     0.900
2     Random Under-Sampling      0.419   0.857     0.562
